## Data Processing [Not Runnable]

In [ ]:
import json
from os.path import join as pj

data_folder = "data/json"
file_path = "hau_adjudicated.json"

json_data = json.load(open(pj(data_folder, file_path)))

english = json_data[:120]
print(len(english))

language = json_data[120:]
print(len(language))

language[0]

In [3]:
t = "wanne irin kan soket zan buƙata a  Niger?"
t[35], t[40], t[35:40]

('N', '?', 'Niger')

In [4]:
import json
from collections import Counter
from typing import List, Dict, Any
from collections import Counter, defaultdict

sptoken = "|"


def compare_entity(e1, e2):
    # 'end': 40, 'text': 'Niger', 'start': 35, 'labels': ['COUNTRY']
    if isinstance(e1, str):
        e1 = decode_entity(e1)
    if isinstance(e2, str):
        e2 = decode_entity(e2)
    return (
        e1["start"] == e2["start"]
        and e1["end"] == e2["end"]
        and e1["labels"] == e2["labels"]
    )


def encode_entity(entity):
    return f"{entity['start']}{sptoken}{entity['end']}{sptoken}{entity['labels'][0]}{sptoken}{entity['text']}"


def decode_entity(entity_str):
    start, end, label, text = entity_str.split(sptoken, 3)
    return {
        "start": int(start),
        "end": int(end),
        "labels": [label],
        "text": text,  # .strip()
    }


def check_overlap(entity1, entity2):
    """Check if two entities overlap based on their start and end positions."""
    if isinstance(entity1, str):
        entity1 = decode_entity(entity1)
    if isinstance(entity2, str):
        entity2 = decode_entity(entity2)
    return (
        entity1["start"] < entity2["end"] and entity1["end"] > entity2["start"]
    ) or (entity2["start"] < entity1["end"] and entity2["end"] > entity1["start"])


def clean_entity(e):
    return {
        "id": e["id"],
        "result": [
            [anno["id"]] + [r["value"] for r in anno["result"]]
            for anno in e["annotations"]
        ],
        "agreement": e["agreement"],
        "text": e["data"]["text"],
        "inner_id": e["inner_id"],
        "total_annotations": e["total_annotations"],
    }


def show_entity(e):
    pp(e)


def analyze_data(data: List[Dict[str, Any]]):
    total_entities = 0
    total_tokens = 0
    all_entities = []
    all_tokens = set()
    entity_distribution = Counter()
    total_disagreements = 0
    annotator_errors = defaultdict(int)
    annotator_total = defaultdict(int)
    disagreed_items = []
    # majority_vote_results = []

    for item in data:
        text = item["data"]["text"]
        tokens = text.split()
        total_tokens += len(tokens)
        all_tokens.update(tokens)

        annotations = item["annotations"]

        # Collect all entities from all annotators
        item_entities = []
        annotator_entities = defaultdict(list)
        for annotation in annotations:
            annotator_id = annotation["completed_by"]["id"]
            entities = [
                encode_entity(result["value"]) for result in annotation["result"]
            ]
            item_entities.extend(entities)
            all_entities.extend(entities)
            annotator_entities[annotator_id] = entities
            annotator_total[annotator_id] += len(entities)

        # Perform majority voting
        entity_counts = Counter(item_entities)
        majority_threshold = len(annotations) / 2
        majority_entities = [
            entity
            for entity, count in entity_counts.items()
            if count > majority_threshold
        ]

        total_entities += len(majority_entities)
        entity_distribution.update(majority_entities)

        # Calculate disagreements and errors
        disagreed_entities = []
        for entity, count in entity_counts.items():
            # if count <= majority_threshold:
            if count <= majority_threshold:

                total_disagreements += 1
                for annotator_id, annotator_ents in annotator_entities.items():
                    if (entity in annotator_ents and count <= majority_threshold) or (
                        entity not in annotator_ents and count > majority_threshold
                    ):
                        annotator_errors[annotator_id] += 1
                if not any(
                    check_overlap(entity, maj_entity)
                    for maj_entity in majority_entities
                ):
                    disagreed_entities.append(
                        {
                            "entity": decode_entity(entity),
                            "annotators": [
                                {
                                    "id": annotator_id,
                                    "included": entity in annotator_ents,
                                }
                                for annotator_id, annotator_ents in annotator_entities.items()
                            ],
                        }
                    )

        if disagreed_entities:
            disagreed_items.append(
                {
                    "inner_id": item["inner_id"],
                    "text": text,
                    "entities": disagreed_entities,
                    "agreement": item["agreement"],
                }
            )

        # disagreed_items[-1]['entities'] = disagreed_entities

    num_texts = len(data)
    avg_entities = total_entities / num_texts
    avg_tokens = total_tokens / num_texts
    unique_entities = len(set(all_entities))
    unique_tokens = len(all_tokens)
    disagreement_percentage = (
        (total_disagreements / total_entities) if total_entities > 0 else 0
    )

    # Calculate error rates for each annotator
    annotator_error_rates = {
        annotator_id: (errors / annotator_total[annotator_id])
        for annotator_id, errors in annotator_errors.items()
    }

    return {
        "avg_entities": avg_entities,
        "avg_tokens": avg_tokens,
        "unique_entities": unique_entities,
        "unique_tokens": unique_tokens,
        "entity_distribution": {
            encode_entity(decode_entity(k)): v
            for k, v in dict(entity_distribution).items()
        },
        "disagreement_percentage": disagreement_percentage,
        "annotator_error_rates": annotator_error_rates,
        "disagreed_items": disagreed_items,
        # "majority_vote_results": majority_vote_results
    }


# Analyze the sample data
results = analyze_data(language)
# Print the results
from pprint import pprint as pp

for key, value in results.items():
    # pp(f"{key}: {value}")
    if key == "entity_distribution":
        # for k, v in value.items():
        #     print(f"{k}: {v}")
        continue
    print(key, end=": ")
    pp(value)

avg_entities: 1.244375
avg_tokens: 9.480625
unique_entities: 3229
unique_tokens: 2329
disagreement_percentage: 0.03088900050226017
annotator_error_rates: {10676: 0.015773660490736103,
 10959: 0.009796533534287867,
 23838: 0.0052763819095477385}
disagreed_items: []


In [8]:
def majority_vote(data: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    all_result = []

    for item in data:
        # Initialize the new item with the original data
        new_item = {
            "id": item["id"],
            "inner_id": item["inner_id"],
            "data": item["data"],
            "total_annotations": item["total_annotations"],
        }
        # Collect all annotations
        all_annotations = []
        for annotation in item["annotations"]:
            for result in annotation["result"]:
                all_annotations.append(
                    {
                        "text": result["value"]["text"],
                        "start": result["value"]["start"],
                        "end": result["value"]["end"],
                        "labels": tuple(
                            result["value"]["labels"]
                        ),  # Convert list to tuple for hashing
                    }
                )

        # Count occurrences of each annotation
        annotation_counts = Counter(tuple(a.items()) for a in all_annotations)

        # Select annotations with majority vote
        majority_threshold = len(item["annotations"]) / 2
        majority_annotations = [
            dict(a)
            for a, count in annotation_counts.items()
            if count > majority_threshold
        ]

        # Convert labels back to list
        for annotation in majority_annotations:
            annotation["labels"] = list(annotation["labels"])

        # Create the new annotation result
        new_annotation = {
            "id": f"majority_vote_{item['id']}",
            "completed_by": {"id": "MAJORITY_VOTE"},
            "result": [
                {
                    "id": f"majority_{i}",
                    "type": "labels",
                    "value": annotation,
                    "origin": "manual",
                    "to_name": "text",
                    "from_name": "label",
                }
                for i, annotation in enumerate(majority_annotations)
            ],
        }

        # Calculate agreement
        total_annotations = sum(annotation_counts.values())
        agreed_annotations = sum(
            count for count in annotation_counts.values() if count > majority_threshold
        )
        agreement = (
            (agreed_annotations / total_annotations) * 100
            if total_annotations > 0
            else 100
        )

        # Add the new annotation and agreement to the item
        new_item["annotations"] = [new_annotation]
        new_item["agreement"] = agreement

        all_result.append(new_item)

    return all_result


from mlds.utils.converter import convert_to_conll2003

lang = "eng"

convert_to_conll2003(majority_vote(english), pj("./data", "conll", f"{lang}.conll"))

120it [00:00, 26725.24it/s]


In [9]:
from mlds.utils.converter import convert_to_conll2003

lang = "hau"
convert_to_conll2003(language, pj("./data", "conll", f"{lang}.conll"))
convert_to_conll2003(
    majority_vote(language), pj("./data", "conll", f"{lang}_majority.conll")
)

9607it [00:00, 26989.66it/s]
3200it [00:00, 21890.94it/s]


In [10]:
# convert to jsonl
# sample output: {"example_id": "test-00003330", "language": "pcm", "text": "Most of de people who dey opposed to Prez Akufo-Addo en decision say within 3 weeks of lockdown, total number of cases for Ghana rise from around 100 catch 1024.", "spans": [{"start_byte": 42, "limit_byte": 52, "label": "PER"}, {"start_byte": 76, "limit_byte": 83, "label": "DATE"}, {"start_byte": 123, "limit_byte": 128, "label": "LOC"}], "target": "PER: Akufo-Addo $$ DATE: 3 weeks $$ LOC: Ghana"}


def convert_to_jsonl(data: List[Dict[str, Any]], output_file: str, lang: str):
    with open(output_file, "w") as f:
        for item in data:
            text = item["data"]["text"]
            spans = []
            for annotation in item["annotations"]:
                for result in annotation["result"]:
                    value = result["value"]
                    spans.append(
                        {
                            "start_byte": value["start"],
                            "limit_byte": value["end"],
                            "label": value["labels"][0],
                        }
                    )
            f.write(
                json.dumps(
                    {
                        "example_id": str(item["inner_id"]),
                        "language": lang,
                        "text": text,
                        "spans": spans,
                        "target": " $$ ".join(
                            [
                                f"{s['label']}: {text[s['start_byte']:s['limit_byte']]}"
                                for s in spans
                            ]
                        ),
                    }
                )
                + "\n"
            )


# convert_to_jsonl(english, pj("./data", "jsonl", "eng.jsonl"), "eng")
# convert_to_jsonl(language, pj("./data", "jsonl", "hau.jsonl"), "hau")
convert_to_jsonl(
    majority_vote(language), pj("./data", "jsonl", "hau_majority.jsonl"), "hau"
)

In [11]:
clean_entity(language[0])

{'id': 122174876,
 'result': [[40932812,
   {'end': 40, 'text': 'Niger', 'start': 35, 'labels': ['COUNTRY']},
   {'end': 20, 'text': 'kan soket', 'start': 11, 'labels': ['SHOPPING_ITEM']}],
  [40955949,
   {'end': 40, 'text': 'Niger', 'start': 35, 'labels': ['COUNTRY']},
   {'end': 20, 'text': 'kan soket', 'start': 11, 'labels': ['SHOPPING_ITEM']}],
  [40928216,
   {'end': 40, 'text': 'Niger', 'start': 35, 'labels': ['COUNTRY']},
   {'end': 20,
    'text': 'kan soket',
    'start': 11,
    'labels': ['SHOPPING_ITEM']}]],
 'agreement': 100.0,
 'text': 'wanne irin kan soket zan buƙata a  Niger?',
 'inner_id': 121,
 'total_annotations': 3}

In [13]:
len(set([clean_entity(x)["text"] for x in language]))  # match them

3200